# PVGIS ST-GNN — MC-Dropout uncertainty pipeline

Reproducible orchestrator for the enhanced MC-Dropout ablation.
It does **not** duplicate runner logic — it builds commands for
`physiq_pv.experiments.pvgis_stgnn_runner` (via the thin
`scripts/run_pvgis_stgnn_forecasting.py` wrapper) and reads the CSVs it writes.

- **Single MC-Dropout run** → logs the run to W&B.
- **Ensemble** → a W&B sweep where the **only** swept parameter is the seed.
- The runner never uploads an outputs artifact to W&B (metrics/summary only).

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.environ['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.chdir(REPO_ROOT)
print('repo root:', REPO_ROOT)

## 2. Shared config

One config feeds both the single run and every ensemble member. MC-Dropout is
always on here (`--mc-dropout`); the predictive interval is the empirical
quantile band of the MC samples.

In [ ]:
CONFIG = dict(
    pvgis_dir='/data/SentinelPV/pvgis_data/data/pvgis_summed_irradiance',
    train_years='2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018',
    test_year=2019,
    anomaly_scores='outputs/pvgis_anomaly_2019_2005_2018_w15_q0975/pvgis_climatology_scores.csv',
    target_variable='pv_power_output',
    model_type='stgnn',
    feature_set='full',
    seq_len=24,
    horizon=1,
    epochs=60,
    batch_size=8,
    dropout=0.3,
    mc_samples=30,
    device='cuda',
    wandb_project='PhysiQ-PV',
    wandb_entity='albertopedalino-politecnico-di-torino',
)

def runner_shared_args(cfg):
    """Fixed runner args shared by the single run and every sweep member.
    Excludes --seed / --out-dir / --wandb* (set per run)."""
    return [
        '--pvgis-dir', cfg['pvgis_dir'],
        '--train-years', cfg['train_years'],
        '--test-year', str(cfg['test_year']),
        '--anomaly-scores', cfg['anomaly_scores'],
        '--target-variable', cfg['target_variable'],
        '--model-type', cfg['model_type'],
        '--feature-set', cfg['feature_set'],
        '--seq-len', str(cfg['seq_len']),
        '--horizon', str(cfg['horizon']),
        '--epochs', str(cfg['epochs']),
        '--batch-size', str(cfg['batch_size']),
        '--dropout', str(cfg['dropout']),
        '--mc-dropout',
        '--mc-samples', str(cfg['mc_samples']),
        '--device', cfg['device'],
        '--wandb-project', cfg['wandb_project'],
        '--wandb-entity', cfg['wandb_entity'],
    ]

checks = {
    'pvgis dir': Path(CONFIG['pvgis_dir']).is_dir(),
    'anomaly scores': Path(CONFIG['anomaly_scores']).exists(),
    'single-run wrapper': Path('scripts/run_pvgis_stgnn_forecasting.py').exists(),
    'sweep member': Path('scripts/run_pvgis_stgnn_sweep_member.py').exists(),
}
try:
    import physiq_pv.experiments.pvgis_stgnn_runner  # noqa: F401
    checks['runner importable'] = True
except Exception as e:
    checks['runner importable'] = False
    print('runner import error:', e)
for k, v in checks.items():
    print(('OK     ' if v else 'MISSING') + '  ' + k)

## 3. Single MC-Dropout run

Logs the run to W&B (metrics + summary). No artifact upload.
Set `RUN_SINGLE = True` to actually launch.

In [ ]:
SEED = 1
SINGLE_OUT_DIR = f'outputs/pvgis_stgnn_mc_dropout/seed{SEED}'

single_cmd = [
    sys.executable, 'scripts/run_pvgis_stgnn_forecasting.py',
    *runner_shared_args(CONFIG),
    '--seed', str(SEED),
    '--out-dir', SINGLE_OUT_DIR,
    '--wandb',
    '--wandb-run-name', f'mc_dropout_seed{SEED}',
]
print(' '.join(single_cmd))
if Path(SINGLE_OUT_DIR).exists():
    print('WARNING: out_dir already exists — a run would overwrite its files.')

In [ ]:
RUN_SINGLE = False
if RUN_SINGLE:
    subprocess.run(single_cmd, check=True)
else:
    print('RUN_SINGLE is False — not launching. Command above is what would run.')

## 4. Ensemble = W&B sweep over seeds only

A real W&B sweep whose **only** parameter is `seed`. Each member
(`scripts/run_pvgis_stgnn_sweep_member.py`) derives a per-seed out-dir and
ensemble-id, then runs the runner with `--wandb`. The runner owns the W&B
run (auto-joins the sweep) and uploads no artifact.

In [ ]:
SEEDS = [1, 2, 3, 4, 5]
ENSEMBLE_OUT_ROOT = 'outputs/pvgis_stgnn_ensemble/runs'
ENSEMBLE_PRED_DIR = 'outputs/pvgis_stgnn_ensemble/predictions'

sweep_config = {
    'program': 'scripts/run_pvgis_stgnn_sweep_member.py',
    'method': 'grid',
    # Informational for a grid sweep; 'mae/global' is a real logged key.
    'metric': {'name': 'mae/global', 'goal': 'minimize'},
    'parameters': {'seed': {'values': SEEDS}},
    'command': [
        '${env}', '${interpreter}', '${program}',
        *runner_shared_args(CONFIG),
        '--out-root', ENSEMBLE_OUT_ROOT,
        '--ensemble-dir', ENSEMBLE_PRED_DIR,
        '${args}',   # wandb appends --seed=<value>; nothing else varies
    ],
}
import json
print(json.dumps(sweep_config, indent=2))

In [ ]:
CREATE_SWEEP = False   # True -> register the sweep on W&B
RUN_AGENT = False      # True -> also run the agent in-process (blocks until done)

if CREATE_SWEEP:
    import wandb
    sweep_id = wandb.sweep(sweep_config, project=CONFIG['wandb_project'],
                           entity=CONFIG['wandb_entity'])
    agent_ref = f"{CONFIG['wandb_entity']}/{CONFIG['wandb_project']}/{sweep_id}"
    print('sweep_id  :', sweep_id)
    print('run with  : wandb agent ' + agent_ref)
    if RUN_AGENT:
        wandb.agent(sweep_id, project=CONFIG['wandb_project'],
                    entity=CONFIG['wandb_entity'], count=len(SEEDS))
else:
    print('Set CREATE_SWEEP=True to register. Program run by each agent worker:')
    print('  ' + sweep_config['program'])
    print('  wandb agent ' + CONFIG['wandb_entity'] + '/' + CONFIG['wandb_project'] + '/<sweep_id>')

## 5. Results (single run)

Reads the CSVs the runner wrote for the single MC-Dropout run.

In [ ]:
def _read(name, out_dir=SINGLE_OUT_DIR):
    p = Path(out_dir) / name
    return pd.read_csv(p) if p.exists() else None

for name in ['metrics.json', 'report.md', 'predictions.csv']:
    p = Path(SINGLE_OUT_DIR) / name
    print(('OK  ' if p.exists() else '--  ') + name)

metrics = _read('metrics_global.csv')
if metrics is not None:
    display(metrics)
else:
    print('No metrics_global.csv yet — run the single MC-Dropout run first.')